

Reads the consolidated dataset (`data/fusion_controls.csv`) into a DF

In [132]:
import os
import pandas as pd

pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)

# resolve the dataset whether the notebook is launched from v2/ or the repo root
path = next(p for p in ['data/fusion_controls.csv', 'v2/data/fusion_controls.csv'] if os.path.exists(p))
df = pd.read_csv(path)
print(f'loaded {len(df)} candidates from {path}')
df['category'].value_counts()

loaded 19 candidates from data/fusion_controls.csv


category
clinical_fusion        9
positive_control       4
secondary_candidate    2
rejected_candidate     2
negative_control       1
worked_case            1
Name: count, dtype: int64

Keep only the positive controls + the single negative control

In [133]:
controls = df[df['category'].isin(['positive_control', 'negative_control'])].reset_index(drop=True)
n_pos = (controls['category'] == 'positive_control').sum()
n_neg = (controls['category'] == 'negative_control').sum()
print(f'{len(controls)} controls  ->  {n_pos} positive + {n_neg} negative')
controls['name'].tolist()

5 controls  ->  4 positive + 1 negative


['GATA2-MECOM', 'TCR-TAL1', 'ETV6-MNX1', 'MIPOL1-ETV1', 'EGFR-SEPT14']

### The runnable view (what the pipeline needs)

In [134]:
RUN_COLS = ['name','category','mechanism','driven_gene','driven_gene_TSS_hg38','strand',
            'element_coords_hg38','alphagenome_ontology']
controls[RUN_COLS]

,name,category,mechanism,driven_gene,driven_gene_TSS_hg38,strand,element_coords_hg38,alphagenome_ontology
0,GATA2-MECOM,positive_control,enhancer_hijacking,EVI1 (MECOM),chr3:169146305,-,chr3:128603548-128604548,EFO:0002067
1,TCR-TAL1,positive_control,enhancer_hijacking (TCR translocation),TAL1,chr1:47232335,-,chr14:22460450-22460810,CL:0000624
2,ETV6-MNX1,positive_control,enhancer_hijacking,MNX1,chr7:157010663,-,chr12:11798088-12011644,CL:0001059
3,MIPOL1-ETV1,positive_control,enhancer_hijacking (translocation),ETV1,chr7:13989666,-,chr14:37246515-37246652,UBERON:0002367
4,EGFR-SEPT14,negative_control,kinase_fusion (promoter_donation),SEPT14 (3') / EGFR kinase,NaN,NaN,NaN,UBERON:0000955


### Full control rows (with provenance)

In [135]:
controls

,name,category,mechanism,disease,driven_gene,driven_gene_TSS_hg38,strand,regulatory_element,element_coords_hg38,fusion_5p_gene,fusion_3p_gene,breakpoint_5p,breakpoint_3p,breakpoint_assembly,structural_event,cell_context,alphagenome_ontology,wetlab_status,paper,pmid,model_result,coord_source,notes
0,GATA2-MECOM,positive_control,enhancer_hijacking,AML inv(3)/t(3;3),EVI1 (MECOM),chr3:169146305,-,G2DHE (GATA2 distal enhancer),chr3:128603548-128604548,NaN,NaN,NaN,NaN,NaN,inv(3)(q21q26)/t(3;3)(q21;q26),"K562 (GATA2+, EVI1-silent)",EFO:0002067,CRISPR_confirmed,"Groschel et al., Cell 2014",24703711,CAPTURED (+0.093 der-WT; enhancer-specific; di...,EVI1 TSS=Ensembl ENST00000264674; G2DHE=Grosch...,The anchor positive control. NOT a fusion gene...
1,TCR-TAL1,positive_control,enhancer_hijacking (TCR translocation),T-ALL t(1;14),TAL1,chr1:47232335,-,TCRδ (Eδ) enhancer,chr14:22460450-22460810,NaN,NaN,NaN,NaN,NaN,t(1;14)(p32;q11),CD4 T-cell (TAL1-silent),CL:0000624,associative (recurrent t(1;14); no CRISPR of T...,Begley PNAS 1989; Chen EMBO J 1990; Eδ: Redond...,2467296; 2303035; 2156339,PENDING RE-RUN (prior +0.116@5kb invalid: buil...,TAL1 (a.k.a. SCL/TCL5) TSS=NCBI Gene 6886; Eδ=...,Gene TAL1=SCL=TCL5. Hijacked element is Eδ (ea...
2,ETV6-MNX1,positive_control,enhancer_hijacking,infant AML t(7;12),MNX1,chr7:157010663,-,ETV6-region hematopoietic enhancer block,chr12:11798088-12011644,NaN,NaN,NaN,NaN,NaN,t(7;12)(q36;p13) (interchromosomal),CD34+ CMP (MNX1-silent),CL:0001059,CRISPR_confirmed (deltaEn block),"Naranjo/Heim et al., Blood Adv 2024",39121370,WEAK (+0.012 der-WT; only block-start window; ...,MNX1 TSS=Ensembl ENST00000252971; enhancer blo...,213 kb block; active ~1kb sub-element unresolv...
3,MIPOL1-ETV1,positive_control,enhancer_hijacking (translocation),prostate adenocarcinoma (LNCaP),ETV1,chr7:13989666,-,chr14q13 enhancer (MIPOL1-intronic; M2 DNase p...,chr14:37246515-37246652,NaN,NaN,NaN,NaN,NaN,t(7;14)(p21;q13),LNCaP prostate adenoca (ETV1-low in normal pro...,UBERON:0002367,CRISPR_confirmed (enhancer deletion: M2 -80.5%...,"Wang et al., Nat Methods 2021",34092790,PENDING (new control replacing LMO2; AlphaGeno...,ETV1 TSS=Ensembl ENST00000430479 / NM_004956.5...,"Genuine enhancer hijacking, NOT a fusion (pape..."
4,EGFR-SEPT14,negative_control,kinase_fusion (promoter_donation),glioblastoma,SEPT14 (3') / EGFR kinase,NaN,NaN,EGFR promoter (donated),NaN,EGFR,SEPTIN14,chr7:55268106:+,chr7:55863785:-,hg19,intrachromosomal del/fusion (~596 kb),brain/GBM,UBERON:0000955,mechanism: chimeric kinase -> STAT3 (trans),"Frattini et al., Nat Genet 2013",23917401,NULL (negative control: promoter donation only...,FusionGDB/ChimerDB4 hg19 (TCGA-06-0750-01A),THE negative control. Kinase fusion - oncogeni...


# Build the actual sequences  

The model needs a **DNA string**, not coordinates. For each control we `fetch()` the reference genome (UCSC REST API — no download) and build the **WT** and **der** contigs, then store them in the DataFrame.
- **enhancer-hijacking** rows: WT = the ~1 Mb window around the oncogene TSS; der = that window with the enhancer inserted `d` kb away (here `d`=10 kb as a demo default; the real analysis sweeps it).
- the **fusion** (negative-control) row: stitched from its breakpoints (`build_fusion`).

In [136]:
import os, sys, importlib
for p in ['..', '.']:
    if os.path.isdir(os.path.join(p,'fusionseq')): sys.path.insert(0, os.path.abspath(p)); break
# Reload so edits to fusionseq/*.py are picked up even in a long-running kernel. (A stale hijack.py
# import -- one without the newer build_der(anchor=...) kwarg -- makes the call below raise, which the
# try/except silently turns into build_error, leaving enh coords None and crashing a later cell.)
import fusionseq.sequences, fusionseq.hijack, fusionseq.pipeline
for _m in (fusionseq.sequences, fusionseq.hijack, fusionseq.pipeline): importlib.reload(_m)
from fusionseq import hijack as hj, pipeline as fp
import pandas as pd

W = 1_048_576          # AlphaGenome window
D_KB = 10              # demo default: enhancer -> TSS distance (real analysis sweeps this)

# Some "enhancers" are annotated as megabase-scale BLOCKS, not ~kb elements. A CENTERED transplant of a
# block wider than 2*D_KB runs over the promoter at the window centre, and a small "proxy" sub-element
# just tests an arbitrary corner of the block. Instead we keep the FULL block and EDGE-anchor it: the
# proximal edge sits D_KB from the TSS and the block extends AWAY from the promoter, so the promoter
# stays intact. The TSS frame additionally SWEEPS the distance for these (see the TSS_centered_data
# cell) because the active sub-element's position in the block is unresolved and these models decay with
# distance, so one placement risks a distance-driven false null. The full block also lives in
# event_site_centered (the faithful derivative). Provenance: data/sources.md.
BLOCK_FULL = {
    # ETV6-MNX1: ETV6 enhancer block = chr12:11,798,088-12,011,644 (213 kb; active sub-element unresolved)
    'ETV6-MNX1': ('chr12', 11_798_088, 12_011_644),
}

def _locus(s):  c,p=s.split(':'); return c, int(p)
def _region(s): c,r=s.split(':'); a,b=r.split('-'); return c, int(a), int(b)

def build_contigs(row):
    out = dict(wt_seq=None, der_seq=None, elem_chrom=None, elem_start=None, elem_end=None,
               element_len=None, insert_kb_from_TSS=None, junction_or_TSS_at=None,
               element_note=None, build_error=None)
    try:
        if 'enhancer_hijacking' in row['mechanism']:
            chrom, tss = _locus(row['driven_gene_TSS_hg38']); ec, es, ee = _region(row['element_coords_hg38'])
            block = row['name'] in BLOCK_FULL
            if block:
                ec, es, ee = BLOCK_FULL[row['name']]            # full block, not a proxy
            wt = hj.build_wt(chrom, tss, W)
            der, off = hj.build_der(chrom, tss, ec, es, ee, d_kb=D_KB, window=W, side='high',
                                    anchor='edge' if block else 'center')
            out.update(wt_seq=wt, der_seq=der, elem_chrom=ec, elem_start=es, elem_end=ee,
                       element_len=ee-es, insert_kb_from_TSS=D_KB, junction_or_TSS_at=W//2)
            if block:
                out['element_note'] = f'FULL block ({ee-es} bp) edge-anchored {D_KB}kb from TSS, extends away'
        else:  # fusion gene -> stitch from breakpoints
            bp5 = int(row['breakpoint_5p'].split(':')[1]); bp3 = int(row['breakpoint_3p'].split(':')[1])
            b = fp.build_fusion(row['fusion_5p_gene'], row['fusion_3p_gene'], bp5, bp3,
                                assembly=row['breakpoint_assembly'], window=W)
            g5, g3 = row['fusion_5p_gene'], row['fusion_3p_gene']
            out.update(wt_seq=b['runs'][f'WT_{g5}'], der_seq=b['runs'][f'der_{g5}__{g3}'], junction_or_TSS_at=W//2)
    except Exception as e:
        out['build_error'] = str(e)[:140]
    return out

print(f'fetching reference + building contigs for {len(controls)} controls (UCSC REST; ~1-2 min)...')
built = controls.apply(lambda r: pd.Series(build_contigs(r)), axis=1)
controls = controls.join(built)
_err = controls['build_error'].notna()
print('done; build errors:', int(_err.sum()))
if _err.any():
    print('!! build FAILED for these rows (fix before continuing -- often a stale import; restart kernel):')
    print(controls.loc[_err, ['name', 'build_error']].to_string(index=False))

fetching reference + building contigs for 5 controls (UCSC REST; ~1-2 min)...
done; build errors: 0


### Sequences are now in the DataFrame — derived info + preview
Full sequences live in `controls['wt_seq']` / `controls['der_seq']` 

In [137]:
def gc(s): return round(100*(s.count('G')+s.count('C'))/len(s), 1) if isinstance(s,str) else None
controls['seq_len']        = controls['der_seq'].map(lambda s: len(s) if isinstance(s,str) else None)
controls['der_gc_pct']     = controls['der_seq'].map(gc)
controls['bp_changed_vs_WT'] = controls.apply(
    lambda r: sum(a!=b for a,b in zip(r['wt_seq'], r['der_seq'])) if isinstance(r['der_seq'],str) else None, axis=1)
controls['der_preview']    = controls['der_seq'].map(lambda s: (s[:40]+' … '+s[-20:]) if isinstance(s,str) else None)
controls[['name','driven_gene','mechanism','seq_len','element_len','insert_kb_from_TSS',
          'bp_changed_vs_WT','der_gc_pct','der_preview','build_error']]

,name,driven_gene,mechanism,seq_len,element_len,insert_kb_from_TSS,bp_changed_vs_WT,der_gc_pct,der_preview,build_error
0,GATA2-MECOM,EVI1 (MECOM),enhancer_hijacking,1048576,1000.0,10.0,783,36.9,CTTACTCTTGCCCACCACTCATGGTGTAGACTGGTCATTT … AGA...,None
1,TCR-TAL1,TAL1,enhancer_hijacking (TCR translocation),1048576,360.0,10.0,262,44.3,TCCACTAAAAAGACAAAATAAGAACAATCATAACTAGCAT … CCT...,None
2,ETV6-MNX1,MNX1,enhancer_hijacking,1048576,213556.0,10.0,159485,44.9,GCCCAGACAACATCATGGATAGAATACAGAGGCTACAATA … TAC...,None
3,MIPOL1-ETV1,ETV1,enhancer_hijacking (translocation),1048576,137.0,10.0,96,35.8,ATTTATTGAGCCTTTCTAGGGTTGTTTCAGGAAGACCTTT … TTT...,None
4,EGFR-SEPT14,SEPT14 (3') / EGFR kinase,kinase_fusion (promoter_donation),1048576,NaN,NaN,391179,43.0,TCCCCAGAGGATTGTTTTGAAATACAGCACTATTGCCCTT … TTA...,None


In [138]:
controls.columns

Index(['name', 'category', 'mechanism', 'disease', 'driven_gene', 'driven_gene_TSS_hg38', 'strand', 'regulatory_element', 'element_coords_hg38', 'fusion_5p_gene', 'fusion_3p_gene', 'breakpoint_5p',
       'breakpoint_3p', 'breakpoint_assembly', 'structural_event', 'cell_context', 'alphagenome_ontology', 'wetlab_status', 'paper', 'pmid', 'model_result', 'coord_source', 'notes', 'wt_seq',
       'der_seq', 'elem_chrom', 'elem_start', 'elem_end', 'element_len', 'insert_kb_from_TSS', 'junction_or_TSS_at', 'element_note', 'build_error', 'seq_len', 'der_gc_pct', 'bp_changed_vs_WT',
       'der_preview'],
      dtype='object')

## `TSS_centered_data` — the clean, ready-to-work-with frame

One tidy DataFrame: coordinates already **parsed** into columns (no more `split(':')`), the sequences, and the readout context — everything the predict/score step needs, nothing it doesn't.

In [139]:
def _loc(s):
    return tuple(_locus(s)) if isinstance(s,str) and s.count(':')==1 else (None,None)

std = controls.copy()
std[['onco_chrom','onco_tss']] = [list(_loc(s)) for s in std['driven_gene_TSS_hg38']]
# enhancer coords = the element ACTUALLY inserted. kb-scale elements: the element itself. Megabase
# BLOCKS (BLOCK_FULL): the FULL block, edge-anchored (see element_note). event_site_centered keeps the
# full block too.
std = std.rename(columns={'elem_chrom':'enh_chrom','elem_start':'enh_start','elem_end':'enh_end',
                          'category':'type','regulatory_element':'enhancer','alphagenome_ontology':'ontology'})

TSS_centered_data = (std
    [['name','type','mechanism','driven_gene','onco_chrom','onco_tss','strand',
      'enhancer','enh_chrom','enh_start','enh_end','element_len','element_note','insert_kb_from_TSS',
      'ontology','cell_context','seq_len','wt_seq','der_seq']]
    .copy())
TSS_centered_data['type'] = TSS_centered_data['type'].str.replace('_control','')

# Distance SWEEP for megabase BLOCKS: the active sub-element's position within the block is unresolved
# and enhancer->promoter signal decays with distance, so a single placement risks a distance-driven
# false null. Replace the single block row with one EDGE-anchored row per distance (full block, promoter
# intact at every distance). Other controls are kb-scale -> left as a single row.
MNX1_SWEEP_KB = [10, 30, 60, 100]
for nm in list(BLOCK_FULL):
    blk = TSS_centered_data[TSS_centered_data['name'] == nm]
    if blk.empty: continue
    base = blk.iloc[0]
    if pd.isna(base['onco_tss']) or pd.isna(base['enh_start']) or pd.isna(base['enh_end']):
        raise ValueError(f"{nm}: no built coords (onco_tss/enh_start/enh_end is NA) -- the contig-build "
                         f"cell failed for it (see its 'build errors' print; usually a stale fusionseq "
                         f"import). Restart the kernel and run from the top.")
    chrom, tss = base['onco_chrom'], int(base['onco_tss'])
    ec, es, ee = base['enh_chrom'], int(base['enh_start']), int(base['enh_end'])
    sweep = []
    for d in MNX1_SWEEP_KB:
        r = base.copy()
        der, _ = hj.build_der(chrom, tss, ec, es, ee, d_kb=d, window=W, side='high', anchor='edge')
        r['name'] = f'{nm}@{d}kb'; r['der_seq'] = der; r['insert_kb_from_TSS'] = d
        r['element_note'] = f'FULL block ({ee-es} bp) edge-anchored {d}kb from TSS, extends away (sweep)'
        sweep.append(r)
    TSS_centered_data = pd.concat([TSS_centered_data[TSS_centered_data['name'] != nm], pd.DataFrame(sweep)],
                                  ignore_index=True)

TSS_centered_data['onco_tss'] = TSS_centered_data['onco_tss'].astype('Int64')
for c in ['enh_start','enh_end','element_len','seq_len']:
    TSS_centered_data[c] = TSS_centered_data[c].astype('Int64')

print('TSS_centered_data:', TSS_centered_data.shape, '| sequences in wt_seq / der_seq')
TSS_centered_data.drop(columns=['wt_seq','der_seq'])

TSS_centered_data: (8, 19) | sequences in wt_seq / der_seq


,name,type,mechanism,driven_gene,onco_chrom,onco_tss,strand,enhancer,enh_chrom,enh_start,enh_end,element_len,element_note,insert_kb_from_TSS,ontology,cell_context,seq_len
0,GATA2-MECOM,positive,enhancer_hijacking,EVI1 (MECOM),chr3,169146305,-,G2DHE (GATA2 distal enhancer),chr3,128603548,128604548,1000,None,10.0,EFO:0002067,"K562 (GATA2+, EVI1-silent)",1048576
1,TCR-TAL1,positive,enhancer_hijacking (TCR translocation),TAL1,chr1,47232335,-,TCRδ (Eδ) enhancer,chr14,22460450,22460810,360,None,10.0,CL:0000624,CD4 T-cell (TAL1-silent),1048576
2,MIPOL1-ETV1,positive,enhancer_hijacking (translocation),ETV1,chr7,13989666,-,chr14q13 enhancer (MIPOL1-intronic; M2 DNase p...,chr14,37246515,37246652,137,None,10.0,UBERON:0002367,LNCaP prostate adenoca (ETV1-low in normal pro...,1048576
3,EGFR-SEPT14,negative,kinase_fusion (promoter_donation),SEPT14 (3') / EGFR kinase,None,<NA>,NaN,EGFR promoter (donated),None,<NA>,<NA>,<NA>,None,NaN,UBERON:0000955,brain/GBM,1048576
4,ETV6-MNX1@10kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 10kb from...,10.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576
5,ETV6-MNX1@30kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 30kb from...,30.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576
6,ETV6-MNX1@60kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 60kb from...,60.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576
7,ETV6-MNX1@100kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 100kb fro...,100.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576


In [140]:
TSS_centered_data.columns

Index(['name', 'type', 'mechanism', 'driven_gene', 'onco_chrom', 'onco_tss', 'strand', 'enhancer', 'enh_chrom', 'enh_start', 'enh_end', 'element_len', 'element_note', 'insert_kb_from_TSS',
       'ontology', 'cell_context', 'seq_len', 'wt_seq', 'der_seq'],
      dtype='object')

In [141]:
TSS_centered_data

,name,type,mechanism,driven_gene,onco_chrom,onco_tss,strand,enhancer,enh_chrom,enh_start,enh_end,element_len,element_note,insert_kb_from_TSS,ontology,cell_context,seq_len,wt_seq,der_seq
0,GATA2-MECOM,positive,enhancer_hijacking,EVI1 (MECOM),chr3,169146305,-,G2DHE (GATA2 distal enhancer),chr3,128603548,128604548,1000,None,10.0,EFO:0002067,"K562 (GATA2+, EVI1-silent)",1048576,CTTACTCTTGCCCACCACTCATGGTGTAGACTGGTCATTTGCATCT...,CTTACTCTTGCCCACCACTCATGGTGTAGACTGGTCATTTGCATCT...
1,TCR-TAL1,positive,enhancer_hijacking (TCR translocation),TAL1,chr1,47232335,-,TCRδ (Eδ) enhancer,chr14,22460450,22460810,360,None,10.0,CL:0000624,CD4 T-cell (TAL1-silent),1048576,TCCACTAAAAAGACAAAATAAGAACAATCATAACTAGCATAAAGTG...,TCCACTAAAAAGACAAAATAAGAACAATCATAACTAGCATAAAGTG...
2,MIPOL1-ETV1,positive,enhancer_hijacking (translocation),ETV1,chr7,13989666,-,chr14q13 enhancer (MIPOL1-intronic; M2 DNase p...,chr14,37246515,37246652,137,None,10.0,UBERON:0002367,LNCaP prostate adenoca (ETV1-low in normal pro...,1048576,ATTTATTGAGCCTTTCTAGGGTTGTTTCAGGAAGACCTTTCTGTCA...,ATTTATTGAGCCTTTCTAGGGTTGTTTCAGGAAGACCTTTCTGTCA...
3,EGFR-SEPT14,negative,kinase_fusion (promoter_donation),SEPT14 (3') / EGFR kinase,None,<NA>,NaN,EGFR promoter (donated),None,<NA>,<NA>,<NA>,None,NaN,UBERON:0000955,brain/GBM,1048576,TCCCCAGAGGATTGTTTTGAAATACAGCACTATTGCCCTTTCGCCA...,TCCCCAGAGGATTGTTTTGAAATACAGCACTATTGCCCTTTCGCCA...
4,ETV6-MNX1@10kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 10kb from...,10.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576,GCCCAGACAACATCATGGATAGAATACAGAGGCTACAATACAGAGA...,GCCCAGACAACATCATGGATAGAATACAGAGGCTACAATACAGAGA...
5,ETV6-MNX1@30kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 30kb from...,30.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576,GCCCAGACAACATCATGGATAGAATACAGAGGCTACAATACAGAGA...,GCCCAGACAACATCATGGATAGAATACAGAGGCTACAATACAGAGA...
6,ETV6-MNX1@60kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 60kb from...,60.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576,GCCCAGACAACATCATGGATAGAATACAGAGGCTACAATACAGAGA...,GCCCAGACAACATCATGGATAGAATACAGAGGCTACAATACAGAGA...
7,ETV6-MNX1@100kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 100kb fro...,100.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576,GCCCAGACAACATCATGGATAGAATACAGAGGCTACAATACAGAGA...,GCCCAGACAACATCATGGATAGAATACAGAGGCTACAATACAGAGA...


In [142]:
out = 'v2/data/TSS_centered_data.csv' if os.path.isdir('v2/data') else 'data/TSS_centered_data.csv'
TSS_centered_data.to_csv(out, index=False)
print(f'saved {out}  ({os.path.getsize(out)/1e6:.1f} MB)')

# round-trip check: re-read and confirm sequences survived
chk = pd.read_csv(out)
print('re-read:', chk.shape)
print('der_seq lengths:', chk['der_seq'].dropna().str.len().tolist())

saved data/TSS_centered_data.csv  (16.8 MB)
re-read: (8, 19)
der_seq lengths: [1048576, 1048576, 1048576, 1048576, 1048576, 1048576, 1048576, 1048576]



| case | junction | breakpoint used | orient | TSS→junction | source / assumption |
|---|---|---|---|---|---|
| **EGFR-SEPT14** (neg) | fusion junction | `chr7:55,268,106` / `chr7:55,863,785` (hg19→hg38 liftover) | fwd | at junction | **REAL** — FusionGDB/ChimerDB4, TCGA-06-0750-01A |
| **GATA2-MECOM** | inv(3) junction | *no consensus bp*; enhancer 40 kb on 3′/low-coord side, **reverse-complemented** | rc | 40 kb | Gröschel *Cell* 2014 (PMID 24703711): inv(3) moves G2DHE to ~30–110 kb 3′ of EVI1, inverted; 40 kb = representative |
| **TCR-TAL1** | t(1;14) junction | *no published bp*; enhancer 20 kb 3′ of TAL1 | fwd | 20 kb *(assumed)* | Begley *PNAS* 1989 (PMID 2467296) + Chen *EMBO J* 1990 (PMID 2303035); TCR-α coords flagged *approx, VERIFY*; distance assumed within the tens-of-kb range typical of TCR translocations |
| **MIPOL1-ETV1** | t(7;14) neo-loop | *approx bp* chr7:~14.15M / chr14:~37.51M | fwd | ~400 kb *(reconstructed)* | Wang *Nat Methods* 2021 (PMID 34092790); chr14q13/MIPOL1-intronic enhancer to intact ETV1; CRISPR-validated; replaces LMO2 |
| **ETV6-MNX1** | t(7;12) junction | *no published bp*; enhancer-block **start** 20 kb 3′ of MNX1 | fwd | 20 kb *(assumed)* | Naranjo *Blood Adv* 2024 (PMID 39121370); enhancer is a 213 kb block, exact sub-element unresolved → block-start used |


In [143]:
from fusionseq import sequences as fl, pipeline as fp

def build_der_junction(onco_chrom, tss, enh_chrom, enh_start, enh_end, d_kb, *, window=W, orient='rc', side='low'):
    '''True derivative centered on the rearrangement junction: oncogene locus on one side,
    heterologous enhancer-bearing partner arm on the other, joined at the breakpoint (window center).
    WT keeps the native oncogene flank on the partner side, so der-WT == the rearrangement.
    Returns (wt, der, readout_bp) where readout_bp is where the oncogene TSS lands.'''
    L = window // 2; d = int(d_kb * 1000)
    if side == 'low':                                            # enhancer 3'/low-coord side of TSS (EVI1-like)
        onco_side    = fl.fetch(onco_chrom, tss - d,     tss - d + L)   # der[L:W] : gene body + TSS
        native_flank = fl.fetch(onco_chrom, tss - d - L, tss - d)       # der[0:L] in WT (no rearrangement)
        partner = (fl.revcomp(fl.fetch(enh_chrom, enh_start, enh_start + L)) if orient == 'rc'
                   else fl.fetch(enh_chrom, enh_end - L, enh_end))       # enhancer adjacent to junction (right end)
        wt, der, readout = native_flank + onco_side, partner + onco_side, L + d
    else:                                                        # enhancer 5'/high-coord side
        onco_side    = fl.fetch(onco_chrom, tss + d - L, tss + d)
        native_flank = fl.fetch(onco_chrom, tss + d,     tss + d + L)
        partner = (fl.revcomp(fl.fetch(enh_chrom, enh_end - L, enh_end)) if orient == 'rc'
                   else fl.fetch(enh_chrom, enh_start, enh_start + L))
        wt, der, readout = onco_side + native_flank, onco_side + partner, L - d
    assert len(wt) == window == len(der), (len(wt), len(der), window)
    return wt, der, readout

# per-case junction geometry (assumed for all but the fusion) -- mirrors the table above
RECON = {
 'GATA2-MECOM': dict(d_kb=40, orient='rc',  side='low', real=False,
                     junction='inv(3) junction; enh +40kb 3\' of EVI1, reverse-complemented',
                     src='Groschel Cell 2014 PMID 24703711 (no consensus bp; 40kb representative)'),
 'TCR-TAL1':    dict(d_kb=20, orient='fwd', side='low', real=False,
                     junction='t(1;14) junction; enh +20kb 3\' of TAL1',
                     src='Begley PNAS 1989 PMID 2467296 / Chen EMBO J 1990 PMID 2303035; enhancer=Edelta Redondo 1990 (M33967); distance assumed'),
 'MIPOL1-ETV1': dict(d_kb=400, orient='fwd', side='low', real=False,
                     junction='t(7;14) neo-loop; chr14q13 enhancer ~400kb from ETV1 (reconstructed)',
                     src='Wang Nat Methods 2021 PMID 34092790; t(7;14) LNCaP enhancer-hijacking (no ETV1 fusion); ~400kb distal'),
 'ETV6-MNX1':   dict(d_kb=20, orient='fwd', side='low', real=False,
                     junction='t(7;12) junction; enh-block start +20kb 3\' of MNX1',
                     src='Naranjo Blood Adv 2024 PMID 39121370; 213kb block, sub-element unresolved'),
}

def _loc(s): 
    if isinstance(s,str) and s.count(':')==1: c,p=s.split(':'); return c,int(p)
    return None,None
def _reg(s):
    if isinstance(s,str) and '-' in s and 'UNRESOLVED' not in s:
        c,r=s.split(':'); a,b=r.split('-'); return c,int(a),int(b)
    return None,None,None

rows=[]
for _,r in controls.iterrows():
    name=r['name']; rec=dict(name=name, type=r['category'].replace('_control',''), mechanism=r['mechanism'],
        driven_gene=r['driven_gene'], strand=r['strand'], enhancer=r['regulatory_element'],
        ontology=r['alphagenome_ontology'], cell_context=r['cell_context'], seq_len=W, build_error=None)
    try:
        if 'enhancer_hijacking' in r['mechanism']:
            g=RECON[name]; oc,tss=_loc(r['driven_gene_TSS_hg38']); ec,es,ee=_reg(r['element_coords_hg38'])
            wt,der,readout=build_der_junction(oc,tss,ec,es,ee,g['d_kb'],orient=g['orient'],side=g['side'])
            rec.update(center_on='inversion/translocation_junction', junction_desc=g['junction'],
                       breakpoint_real=g['real'], breakpoint_source=g['src'],
                       onco_chrom=oc, onco_tss=tss, enh_chrom=ec, enh_start=es, enh_end=ee,
                       element_len=ee-es, tss_to_junction_kb=g['d_kb'], orient=g['orient'],
                       readout_bp=readout, wt_seq=wt, der_seq=der)
        else:                                                    # fusion (real breakpoint, junction-centered)
            bp5=int(r['breakpoint_5p'].split(':')[1]); bp3=int(r['breakpoint_3p'].split(':')[1])
            g5,g3=r['fusion_5p_gene'],r['fusion_3p_gene']
            b=fp.build_fusion(g5,g3,bp5,bp3,assembly=r['breakpoint_assembly'],window=W)
            rec.update(center_on='fusion_junction',
                       junction_desc=f'{g5}:{bp5}|{g3}:{bp3} ({r["breakpoint_assembly"]}->hg38 liftover)',
                       breakpoint_real=True, breakpoint_source=r['coord_source'],
                       onco_chrom=None, onco_tss=pd.NA, enh_chrom=None, enh_start=pd.NA, enh_end=pd.NA,
                       element_len=pd.NA, tss_to_junction_kb=0, orient='fwd',
                       readout_bp=W//2, wt_seq=b['runs'][f'WT_{g5}'], der_seq=b['runs'][f'der_{g5}__{g3}'])
    except Exception as e:
        rec['build_error']=str(e)[:160]
    rows.append(rec)

cols=['name','type','mechanism','driven_gene','center_on','junction_desc','breakpoint_real','breakpoint_source',
      'onco_chrom','onco_tss','strand','enhancer','enh_chrom','enh_start','enh_end','element_len',
      'tss_to_junction_kb','orient','ontology','cell_context','seq_len','readout_bp','wt_seq','der_seq','build_error']
event_site_centered_data=pd.DataFrame(rows)[cols]
for c in ['onco_tss','enh_start','enh_end','element_len','seq_len','readout_bp']:
    event_site_centered_data[c]=event_site_centered_data[c].astype('Int64')
print('event_site_centered_data:', event_site_centered_data.shape, '| build errors:', event_site_centered_data['build_error'].notna().sum())
print(event_site_centered_data[['name','center_on','breakpoint_real','tss_to_junction_kb','orient','readout_bp','seq_len','build_error']].to_string(index=False))
event_site_centered_data.drop(columns=['wt_seq','der_seq','junction_desc','breakpoint_source'])

event_site_centered_data: (5, 25) | build errors: 0
       name                        center_on  breakpoint_real  tss_to_junction_kb orient  readout_bp  seq_len build_error
GATA2-MECOM inversion/translocation_junction            False                  40     rc      564288  1048576        None
   TCR-TAL1 inversion/translocation_junction            False                  20    fwd      544288  1048576        None
  ETV6-MNX1 inversion/translocation_junction            False                  20    fwd      544288  1048576        None
MIPOL1-ETV1 inversion/translocation_junction            False                 400    fwd      924288  1048576        None
EGFR-SEPT14                  fusion_junction             True                   0    fwd      524288  1048576        None


,name,type,mechanism,driven_gene,center_on,breakpoint_real,onco_chrom,onco_tss,strand,enhancer,enh_chrom,enh_start,enh_end,element_len,tss_to_junction_kb,orient,ontology,cell_context,seq_len,readout_bp,build_error
0,GATA2-MECOM,positive,enhancer_hijacking,EVI1 (MECOM),inversion/translocation_junction,False,chr3,169146305,-,G2DHE (GATA2 distal enhancer),chr3,128603548,128604548,1000,40,rc,EFO:0002067,"K562 (GATA2+, EVI1-silent)",1048576,564288,None
1,TCR-TAL1,positive,enhancer_hijacking (TCR translocation),TAL1,inversion/translocation_junction,False,chr1,47232335,-,TCRδ (Eδ) enhancer,chr14,22460450,22460810,360,20,fwd,CL:0000624,CD4 T-cell (TAL1-silent),1048576,544288,None
2,ETV6-MNX1,positive,enhancer_hijacking,MNX1,inversion/translocation_junction,False,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,20,fwd,CL:0001059,CD34+ CMP (MNX1-silent),1048576,544288,None
3,MIPOL1-ETV1,positive,enhancer_hijacking (translocation),ETV1,inversion/translocation_junction,False,chr7,13989666,-,chr14q13 enhancer (MIPOL1-intronic; M2 DNase p...,chr14,37246515,37246652,137,400,fwd,UBERON:0002367,LNCaP prostate adenoca (ETV1-low in normal pro...,1048576,924288,None
4,EGFR-SEPT14,negative,kinase_fusion (promoter_donation),SEPT14 (3') / EGFR kinase,fusion_junction,True,None,<NA>,NaN,EGFR promoter (donated),None,<NA>,<NA>,<NA>,0,fwd,UBERON:0000955,brain/GBM,1048576,524288,None


In [144]:
out2 = 'v2/data/event_site_centered_data.csv' if os.path.isdir('v2/data') else 'data/event_site_centered_data.csv'
event_site_centered_data.to_csv(out2, index=False)
print(f'saved {out2}  ({os.path.getsize(out2)/1e6:.1f} MB)')
chk = pd.read_csv(out2)
print('re-read:', chk.shape, '| der_seq lengths:', chk['der_seq'].dropna().str.len().tolist())

saved data/event_site_centered_data.csv  (10.5 MB)
re-read: (5, 25) | der_seq lengths: [1048576, 1048576, 1048576, 1048576, 1048576]


In [145]:
event_site_centered_data.columns

Index(['name', 'type', 'mechanism', 'driven_gene', 'center_on', 'junction_desc', 'breakpoint_real', 'breakpoint_source', 'onco_chrom', 'onco_tss', 'strand', 'enhancer', 'enh_chrom', 'enh_start',
       'enh_end', 'element_len', 'tss_to_junction_kb', 'orient', 'ontology', 'cell_context', 'seq_len', 'readout_bp', 'wt_seq', 'der_seq', 'build_error'],
      dtype='object')

 `n_padded_TSS_centered_data` — same as TSS-centered, but flanks are N instead of reference
Same as TSS centered but with N padding

In [146]:
CORE_KB = 64   # keep this radius (kb) of REAL sequence around the functional span; N-pad the rest
def _pad(seq, lo, hi, window=W):
    return 'N' * lo + seq[lo:hi] + 'N' * (window - hi)

def keep_span(r):
    '''[lo, hi) of REAL sequence to retain for this row; everything outside is N-padded.
    kb-scale elements: symmetric +-CORE_KB around the promoter (the element sits inside it).
    Megabase BLOCKS (BLOCK_FULL): a symmetric core would N-pad away most of the block, so instead keep
    the promoter THROUGH the full edge-anchored block, + CORE_KB of flank on each outer side -- same
    idea (isolate promoter+element from distal context) but sized to the block. Assumes side="high".'''
    c = W // 2; f = CORE_KB * 1000
    if r['name'].startswith(tuple(BLOCK_FULL)):
        off = c + int(r['insert_kb_from_TSS'] * 1000); w = int(r['element_len'])   # edge-anchored, side='high'
        return max(0, c - f), min(W, off + w + f)
    return c - f, c + f

n_padded_TSS_centered_data = TSS_centered_data.copy()
spans = list(TSS_centered_data.apply(keep_span, axis=1))
n_padded_TSS_centered_data['wt_seq']  = [_pad(s, lo, hi) for s,(lo,hi) in zip(TSS_centered_data['wt_seq'],  spans)]
n_padded_TSS_centered_data['der_seq'] = [_pad(s, lo, hi) for s,(lo,hi) in zip(TSS_centered_data['der_seq'], spans)]
n_padded_TSS_centered_data['core_kb']    = CORE_KB
n_padded_TSS_centered_data['kept_bp']    = [hi - lo for (lo, hi) in spans]
n_padded_TSS_centered_data['n_pad_frac'] = (1 - n_padded_TSS_centered_data['kept_bp'] / W).round(3)

# sanity: length preserved; for every row the kept core is byte-identical to TSS_centered and the
# promoter (center +-2kb) survived (never N), and the distal flanks are all N.
c = W // 2; ok_core = ok_prom = ok_flank = True
for (lo, hi), npd, ref in zip(spans, n_padded_TSS_centered_data['der_seq'], TSS_centered_data['der_seq']):
    ok_core  &= npd[lo:hi] == ref[lo:hi]
    ok_prom  &= 'N' not in npd[c-2000:c+2000]
    ok_flank &= (lo == 0 or set(npd[:lo]) == {'N'}) and (hi == W or set(npd[hi:]) == {'N'})
print(f'len ok: {all(len(s)==W for s in n_padded_TSS_centered_data["der_seq"])} | core identical: {ok_core} | promoter intact: {ok_prom} | flanks all N: {ok_flank}')
print(n_padded_TSS_centered_data[['name','core_kb','kept_bp','n_pad_frac','seq_len']].to_string(index=False))
n_padded_TSS_centered_data.drop(columns=['wt_seq','der_seq'])

len ok: True | core identical: True | promoter intact: True | flanks all N: True
           name  core_kb  kept_bp  n_pad_frac  seq_len
    GATA2-MECOM       64   128000       0.878  1048576
       TCR-TAL1       64   128000       0.878  1048576
    MIPOL1-ETV1       64   128000       0.878  1048576
    EGFR-SEPT14       64   128000       0.878  1048576
 ETV6-MNX1@10kb       64   351556       0.665  1048576
 ETV6-MNX1@30kb       64   371556       0.646  1048576
 ETV6-MNX1@60kb       64   401556       0.617  1048576
ETV6-MNX1@100kb       64   441556       0.579  1048576


,name,type,mechanism,driven_gene,onco_chrom,onco_tss,strand,enhancer,enh_chrom,enh_start,enh_end,element_len,element_note,insert_kb_from_TSS,ontology,cell_context,seq_len,core_kb,kept_bp,n_pad_frac
0,GATA2-MECOM,positive,enhancer_hijacking,EVI1 (MECOM),chr3,169146305,-,G2DHE (GATA2 distal enhancer),chr3,128603548,128604548,1000,None,10.0,EFO:0002067,"K562 (GATA2+, EVI1-silent)",1048576,64,128000,0.878
1,TCR-TAL1,positive,enhancer_hijacking (TCR translocation),TAL1,chr1,47232335,-,TCRδ (Eδ) enhancer,chr14,22460450,22460810,360,None,10.0,CL:0000624,CD4 T-cell (TAL1-silent),1048576,64,128000,0.878
2,MIPOL1-ETV1,positive,enhancer_hijacking (translocation),ETV1,chr7,13989666,-,chr14q13 enhancer (MIPOL1-intronic; M2 DNase p...,chr14,37246515,37246652,137,None,10.0,UBERON:0002367,LNCaP prostate adenoca (ETV1-low in normal pro...,1048576,64,128000,0.878
3,EGFR-SEPT14,negative,kinase_fusion (promoter_donation),SEPT14 (3') / EGFR kinase,None,<NA>,NaN,EGFR promoter (donated),None,<NA>,<NA>,<NA>,None,NaN,UBERON:0000955,brain/GBM,1048576,64,128000,0.878
4,ETV6-MNX1@10kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 10kb from...,10.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576,64,351556,0.665
5,ETV6-MNX1@30kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 30kb from...,30.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576,64,371556,0.646
6,ETV6-MNX1@60kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 60kb from...,60.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576,64,401556,0.617
7,ETV6-MNX1@100kb,positive,enhancer_hijacking,MNX1,chr7,157010663,-,ETV6-region hematopoietic enhancer block,chr12,11798088,12011644,213556,FULL block (213556 bp) edge-anchored 100kb fro...,100.0,CL:0001059,CD34+ CMP (MNX1-silent),1048576,64,441556,0.579


In [ ]:
out3 = 'v2/data/n_padded_TSS_centered_data.csv' if os.path.isdir('v2/data') else 'data/n_padded_TSS_centered_data.csv'
n_padded_TSS_centered_data.to_csv(out3, index=False)
print(f'saved {out3}  ({os.path.getsize(out3)/1e6:.1f} MB)')
chk = pd.read_csv(out3)
print('re-read:', chk.shape, '| N in der_seq[0]:', str(chk['der_seq'].iloc[0]).count('N'),
      '| der_seq lengths:', chk['der_seq'].dropna().str.len().tolist())

saved data/n_padded_TSS_centered_data.csv  (16.8 MB)
re-read: (8, 22) | N in der_seq[0]: 920576 | der_seq lengths: [1048576, 1048576, 1048576, 1048576, 1048576, 1048576, 1048576, 1048576]


: 